# Lab 10 — Gradient Boosting — Life Expectancy Bands
**Classification Track** · Intermediate · ~55 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Bin life expectancy into tercile bands
2. Train GradientBoostingClassifier
3. Read feature importance for global insight
4. Sketch partial dependence on gdpPercap

## Datasets (this folder)
- `gapminderDataFiveYear.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-10-gradient-boosting-gapminder/lab-10-gradient-boosting-gapminder.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`gapminderDataFiveYear.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-10-gradient-boosting-gapminder"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-10-gradient-boosting-gapminder/bundle/dataset.zip"
NEED = ["gapminderDataFiveYear.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Ensemble Methods: Life Expectancy Bands

> **Scenario:** The World Health Organisation wants a lightweight screening tool that classifies countries into Low / Mid / High life-expectancy bands using only GDP per capita, population, and the survey year. You will build a gradient boosting classifier, interrogate its feature importances, and discover why one of your features is a leakage trap.
>
> **You will learn:** tercile binning with `pd.qcut`, GradientBoostingClassifier, feature importances, partial dependence, temporal leakage.
> **Time:** ~55 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental Map

| Section | Key idea | sklearn / pandas tool |
|---|---|---|
| 1 | Turn a regression target into 3 balanced classes | `pd.qcut` |
| 2 | Fit a gradient boosting classifier | `GradientBoostingClassifier` |
| 3 | Read feature importances; spot the year leak | `feature_importances_` |
| 4 | Sketch partial dependence on GDP | `partial_dependence` |

---

## 1. From Regression Target to Banded Classification

Public-health decisions are rarely made on a continuous scale. A ministry of health does not ask "what is the exact life expectancy of this country?" — it asks "is this country on track, falling behind, or leading?" Turning a regression target into a small number of ordered bands is one of the most common reframings a data scientist performs, and it changes everything downstream: which metrics you trust, which errors matter, and how you explain the model to non-technical stakeholders.

`lifeExp` is continuous, but policy dashboards speak in bands. We bin it into **terciles** — three groups with (as near as possible) equal counts — using `pd.qcut`. The key word is *quantile*: `qcut` chooses cut points so each band holds the same number of rows, rather than chopping the value axis into equal-width slices. Think of it like grading on a curve: the bottom third of countries get "Low", the middle third "Mid", and the top third "High", regardless of the actual year values at the boundaries.

This matters because the alternative, `pd.cut`, divides the axis into equal-width intervals. On a skewed target like life expectancy — where most countries cluster between 60 and 80 years — equal-width bins would dump the vast majority of rows into one or two bands and leave the rest nearly empty. Quantile binning guarantees balance, and balance is what makes accuracy a meaningful metric in the next section.

The snippet below loads the Gapminder five-year panel, prints the shape and summary statistics, then applies `pd.qcut` with three quantile-based bins labelled Low / Mid / High. The two prints after binning let you verify the cut points and inspect the per-band min/max ranges, so you can see exactly where the boundaries landed.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("gapminderDataFiveYear.csv")
print(df.shape)
print(df["lifeExp"].describe().round(2))

df["life_band"] = pd.qcut(df["lifeExp"], q=3, labels=["Low", "Mid", "High"])
print(df["life_band"].value_counts().sort_index())
print(df.groupby("life_band", observed=True)["lifeExp"].agg(["min", "max", "count"]).round(2))


**What to notice:**
1. `qcut` cuts at the 33.3rd and 66.7th percentiles, so each band holds exactly 568 of the 1704 rows.
2. The bands are balanced by construction — accuracy is a meaningful metric here (unlike churn data later).
3. `observed=True` in `groupby` keeps the categorical ordering explicit and silence-free.

> **Pitfall:** `pd.cut` (equal-width bins) would put almost everything in one band because `lifeExp` is left-skewed. Always check `value_counts()` after binning.

> **In practice:** The WHO and World Bank routinely bin continuous indicators — life expectancy, child mortality, GDP per capita — into terciles or quartiles for country scorecards. Bands travel better in slide decks than regression coefficients, and they make "which group is this country in?" answerable at a glance. The cost is information loss: two countries one year apart can land in different bands, so always keep the continuous column around for later analysis.

---

## 2. Training the Gradient Boosting Classifier

Gradient boosting builds shallow decision trees **sequentially**, each one correcting the residuals of the ensemble so far:

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X = df[["gdpPercap", "pop", "year"]]
y = df["life_band"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

gbm = GradientBoostingClassifier(random_state=42)
gbm.fit(X_train, y_train)

train_acc = accuracy_score(y_train, gbm.predict(X_train))
test_acc = accuracy_score(y_test, gbm.predict(X_test))
print(f"train accuracy: {train_acc:.4f}")
print(f"test accuracy:  {test_acc:.4f}")


**What to notice:**
1. Default GBM uses 100 trees of depth 3 — already strong on this dataset (train ≈ 0.88, test ≈ 0.75).
2. The train–test gap is the overfitting budget you can tune with `n_estimators`, `max_depth`, `learning_rate`.
3. `stratify=y` keeps the band proportions identical in train and test.

> **Pitfall:** never tune hyperparameters on the test set — use `cross_val_score` or a validation split, or you are measuring noise.

---

## 3. Feature Importance and the Year Leakage Trap

In [ ]:
imp = pd.Series(gbm.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(4))

print(df[["lifeExp", "gdpPercap", "pop", "year"]].corr()["lifeExp"].round(3))


**What to notice:**
1. `gdpPercap` dominates (importance ≈ 0.76) — richer countries live longer, and GDP proxies nutrition, healthcare, sanitation.
2. `year` correlates with `lifeExp` at ≈ +0.44: life expectancy rose almost everywhere over 1952–2007. The model leans on `year` as a cheap proxy for "how modern is this row".
3. That is **temporal leakage** for any real forecasting use: at prediction time you would not know which year a new observation belongs to, and the year→lifeExp drift is a trend, not a law.

> **Pitfall:** a feature can be statistically predictive and still be leakage. Ask: "will I know this value, with this meaning, at prediction time?" If the answer is no, drop it — refit without `year` and watch test accuracy fall from 0.7465 to 0.6925.

---

## 4. Partial Dependence Sketch on GDP

Partial dependence answers: *holding everything else at its observed values, how does the model's output change as `gdpPercap` varies?*

In [ ]:
from sklearn.inspection import partial_dependence

pd_result = partial_dependence(gbm, X_train, features=["gdpPercap"], kind="average")
grid = pd_result["grid_values"][0]
margins = np.asarray(pd_result["average"])
if margins.shape[0] == len(grid):
    margins = margins.T
print("classes_:", list(gbm.classes_))
print("grid points:", len(grid), "| margins shape:", margins.shape)

probs = np.exp(margins - margins.max(axis=0, keepdims=True))
probs = probs / probs.sum(axis=0, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 4))
for i, cls in enumerate(gbm.classes_):
    ax.plot(grid, probs[i], marker="o", ms=3, label=f"P({cls})")
ax.set_xlabel("gdpPercap (GDP per capita, USD)")
ax.set_ylabel("Partial dependence (probability)")
ax.set_title("GBM partial dependence on gdpPercap")
ax.legend()
fig.tight_layout()
fig.savefig("lab10_partial_dependence.png", dpi=120)
from IPython.display import display
display(fig)
print("saved lab10_partial_dependence.png")


**What to notice:**
1. `partial_dependence` on a classifier returns **raw margins** (log-odds-like), not probabilities — the softmax converts them to a readable 0–1 scale.
2. `gbm.classes_` is **alphabetical** (`['High', 'Low', 'Mid']`), not the order you passed to `qcut`. Always check `classes_` before indexing per-class outputs.
3. The High-band probability rises from ≈ 0.004 to ≈ 0.93 across the GDP grid while Low falls from ≈ 0.96 to ≈ 0.02 — the signature of a 3-class problem.

> **Pitfall:** never read a PD curve as causation. GDP per capita is entangled with continent, era, and institutions.

---

## Exercises (do these!)

### Exercise 1 — Find the tercile cut points
Compute the exact `lifeExp` values at the 1/3 and 2/3 quantiles that `pd.qcut` used.
*Expected: q(1/3) ≈ 52.10, q(2/3) ≈ 68.91.*

**Follow-up:** How many rows fall in each band? Check: 568 / 568 / 568.

### Exercise 2 — Honest test accuracy
Refit the GBM **without** `year` (features: `gdpPercap`, `pop`) and report test accuracy.
*Expected: ≈ 0.6925.*

**Follow-up:** How much accuracy did the year leak buy you? Check: ≈ +0.054.

### Exercise 3 — Partial dependence endpoints
Using the PD array from section 4, print the softmax probability for the **High** band at the smallest and largest `gdpPercap` grid points.
*Expected: ≈ 0.004 at the low end, ≈ 0.934 at the high end.*

**Follow-up:** Does the High-band curve rise overall? Check: yes — from 0.004 to 0.934.

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
q13, q23 = df["lifeExp"].quantile([1/3, 2/3])
print(f"tercile cut points: {q13:.4f}, {q23:.4f}")
print(df["life_band"].value_counts().sort_index())

# --- Solution 2 ---
X_honest = df[["gdpPercap", "pop"]]
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
gbm_honest = GradientBoostingClassifier(random_state=42).fit(X_tr, y_tr)
acc_honest = accuracy_score(y_te, gbm_honest.predict(X_te))
print(f"honest test accuracy (no year): {acc_honest:.4f}")
print(f"leak bought: {test_acc - acc_honest:+.4f}")

# --- Solution 3 ---
high_idx = list(gbm.classes_).index("High")
print(f"PD High @ min grid: {probs[high_idx, 0]:.4f}")
print(f"PD High @ max grid: {probs[high_idx, -1]:.4f}")

# --- Follow-up 1 ---
counts = df["life_band"].value_counts().sort_index()
assert abs(counts.min() - counts.max()) <= 1, "bands should be near-equal"
print("band counts:", counts.to_dict())

# --- Follow-up 2 ---
assert acc_honest < test_acc, "year leak should inflate accuracy"
print(f"leak bonus: {test_acc - acc_honest:.4f}")

# --- Follow-up 3 ---
assert probs[high_idx, -1] > probs[high_idx, 0], "High PD should rise with GDP"
print(f"High-band PD rises from {probs[high_idx, 0]:.4f} to {probs[high_idx, -1]:.4f}")


### What to learn next
- Tune GBM with `GridSearchCV` over `learning_rate` × `n_estimators` (the classic trade-off).
- Try `HistGradientBoostingClassifier` on a bigger dataset — same idea, 10× faster.
- Compare against `RandomForestClassifier`: bagging vs boosting.

*Files in this folder: gapminderDataFiveYear.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
